## Project
Using Time Series Analysis To Forecast Future Vacancy Levels 

## Hypothesis
If we examine historical vacancy data with data science techniques,

we can identify key patterns that occur over time and contribute to accurate forecasting, 

to drive actionable insights on labour market tightness for monetary policy makers. 

Why this matters? 

Higher vacancies -> lower unemployment -> higher inflation -> potential interest rate measures 

## Task 
(estimated hours)

1. Automate scraping of source files (0.5)
2. Consolidate data (0.5)
3. Visualise patterns and revisions (0.5)
4. Build a simple forecasting baseline (1)
5. Outline evaluation, recommendations, and next steps. (0.5)

All code (including small helpers) has been included in this notebook for readability. 

# 1. Import libraries and data

o Automate the download (or scraping) of the CSV files on the ONS website with a subset of at least 20 

In [ ]:
# 1a. Import libraries

from __future__ import annotations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
import time
import random
import csv
import re

from pathlib import Path
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from email.utils import parsedate_to_datetime
from datetime import datetime, timezone
from dateutil.parser import parse as dtparse
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.forecasting.stl import STLForecast
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing

In [ ]:
# 1b. Scrape data from the web

PREV_URL = "https://www.ons.gov.uk/employmentandlabourmarket/peopleinwork/employmentandemployeetypes/timeseries/ap2y/lms/previous"
RAW_DIR  = (Path.cwd() / "../data/raw").resolve()
RAW_DIR.mkdir(parents=True, exist_ok=True)
USER_AGENT = "FVL-Interview-Task/1.0 (+https://github.com/<seleklekterek>/fvl-project; contact:<197108180+seleklekterek@users.noreply.github.com>)"

def scrape_ap2y_csv_links(page_url: str) -> list[str]:
    s = requests.Session()
    s.headers.update({"User-Agent": USER_AGENT})
    r = s.get(page_url, timeout=15)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    urls = [
        urljoin(page_url, a["href"])
        for a in soup.select("a[href]")
        if "format=csv" in a["href"].lower()
    ]
    seen, out = set(), []
    for u in urls:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out

def _head_last_modified(s: requests.Session, url: str) -> datetime | None:
    try:
        h = s.head(url, timeout=15, allow_redirects=True)
        if h.status_code >= 400 or "Last-Modified" not in h.headers:
            g = s.get(url, stream=True, timeout=30, allow_redirects=True)
            g.raise_for_status()
            lm = g.headers.get("Last-Modified")
            g.close()
        else:
            lm = h.headers.get("Last-Modified")

        if not lm:
            return None
        dt = parsedate_to_datetime(lm)
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        else:
            dt = dt.astimezone(timezone.utc)
        return dt
    except Exception:
        return None

def _make_dst_name(dt: datetime | None, ix: int) -> str:
    if dt:
        stamp = dt.strftime("%Y%m%d")
        return f"AP2Y_prev_{stamp}.csv"
    else:
        return f"AP2Y_prev_idx{ix:03d}.csv"

def download_csvs_chrono(
    urls: list[str],
    raw_dir: Path,
    limit: int = 20,
    delay_s: float = 0.15,
    max_retries: int = 6,
    backoff_base: float = 2.0,
    order: str = "asc",  
):
    s = requests.Session()
    s.headers.update({"User-Agent": USER_AGENT})
    raw_dir.mkdir(parents=True, exist_ok=True)

    meta = []
    for u in urls:
        lm = _head_last_modified(s, u)
        meta.append((u, lm))

    meta.sort(key=lambda x: (x[1] or datetime.max.replace(tzinfo=timezone.utc)))
    if order == "desc":
        meta.reverse()

    meta = meta[:limit]

    seen_names = set()
    saved = []
    for ix, (url, dt) in enumerate(meta, start=1):
        base = _make_dst_name(dt, ix)
        name = base
        k = 1
        while name in seen_names or (raw_dir / name).exists():
            k += 1
            stem, ext = base.rsplit(".", 1)
            name = f"{stem}__v{k}.{ext}"
        seen_names.add(name)

        dst = raw_dir / name

        if dst.exists() and dst.stat().st_size > 0:
            saved.append(dst)
            continue

        for attempt in range(max_retries):
            try:
                resp = s.get(url, timeout=15, allow_redirects=True)
                if resp.status_code == 429:
                    ra = resp.headers.get("Retry-After")
                    wait = float(ra) + random.uniform(0.2, 0.8) if ra else (backoff_base ** attempt) + random.uniform(0.2, 0.8)
                    time.sleep(wait)
                    continue
                resp.raise_for_status()
                dst.write_bytes(resp.content)
                if dst.stat().st_size > 0:
                    saved.append(dst)
                    time.sleep(delay_s + random.uniform(0, 0.5))
                break
            except requests.RequestException as exc:
                wait = (backoff_base ** attempt) + random.uniform(0.2, 0.8)
                time.sleep(wait)
                if attempt == max_retries - 1:
                    print(f"[warn] {url} -> {exc}")
            except Exception as exc:
                print(f"[warn] {url} -> {exc}")
                break
    return saved

# run
urls = scrape_ap2y_csv_links(PREV_URL)
downloaded = download_csvs_chrono(urls, RAW_DIR, limit=20, order="asc")  
len(downloaded), [p.name for p in downloaded[:5]]

Challenges 
- Couldn't download as many as 20 files in one go - had to amend the code to respect ONS rate limits 
- Upon closer inspection a couple of files from 2019 appeared in the set - had to amend the code to get the first 20 in historical order

Next 
- Optimise speed of retrieval - at over a minute this is too slow (this is because of one extra network round trip per URL to learn dates before donwloading for correct ordering)

# 2. Prepare data

o Clean and consolidate downloaded csvs capturing observation and release dates for each value. 


In [ ]:
# 2a. Consolidate all monthly data

RAW_DIR = (Path.cwd() / "../data/raw").resolve()
PROCESSED_DIR = (Path.cwd() / "../data/processed").resolve()
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

MONTH_MAP = {"JAN":1,"FEB":2,"MAR":3,"APR":4,"MAY":5,"JUN":6,
             "JUL":7,"AUG":8,"SEP":9,"OCT":10,"NOV":11,"DEC":12}

# helpers
def _detect_header_and_data_start(csv_path: Path) -> tuple[dict[str, str], int, bool]:
    header: dict[str, str] = {}
    data_start = 0
    has_header_row = False

    with csv_path.open("r", encoding="utf-8", newline="") as f:
        rows = list(csv.reader(f))

    def _is_numeric(x: str) -> bool:
        try:
            float(x.replace(",", ""))
            return True
        except Exception:
            return False

    period_pat = re.compile(
        r"""^
            \d{4}(
                \s+(JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC) |
                \s+Q[1-4] |
                \s+M(?:[1-9]|1[0-2])
            )?
        $""",
        re.IGNORECASE | re.VERBOSE,
    )

    for i, row in enumerate(rows):
        if not row:
            continue
        first = (row[0] or "").strip().lstrip("\ufeff").lower()

        if first in {"date", "period", "time"}:
            data_start = i
            has_header_row = True
            break

        if len(row) >= 2:
            tok = (row[0] or "").strip()
            val = (row[1] or "").strip()
            if period_pat.match(tok) and _is_numeric(val):
                data_start = i
                has_header_row = False
                break

    for r in rows[:data_start]:
        if len(r) >= 2 and r[0] and r[1]:
            header[str(r[0]).strip().lower()] = str(r[1]).strip()
        elif len(r) == 1 and r[0]:
            txt = str(r[0])
            if ":" in txt:
                k, v = txt.split(":", 1)
                k = k.strip().lower()
                v = v.strip()
                if k:
                    header[k] = v

    return header, data_start, has_header_row

def _extract_vintage_date(header: dict[str, str], fp: Path) -> pd.Timestamp:
    val = header.get("release date")
    if val:
        try:
            return pd.Timestamp(dtparse(val, dayfirst=True).date()) # to ensure it defaults to UK format
        except Exception:
            pass
    head_text = ""
    try:
        with fp.open("r", encoding="utf-8", errors="ignore") as f:
            for _ in range(60):
                try:
                    head_text += next(f)
                except StopIteration:
                    break
    except Exception:
        head_text = ""

    m = re.search(r"release\s*date\s*[:|,]\s*([0-9]{4}-[0-9]{2}-[0-9]{2}|[0-9]{1,2}\s+\w+\s+[0-9]{4}|[0-9]{1,2}[\/\-\.][0-9]{1,2}[\/\-\.][0-9]{4})",
              head_text, flags=re.IGNORECASE)
    if m:
        try:
            return pd.Timestamp(dtparse(m.group(1), dayfirst=True).date())
        except Exception:
            pass

    m2 = re.search(r"(\d{4})(\d{2})(\d{2})", fp.name)
    if m2:
        y, mm, dd = map(int, m2.groups())
        return pd.Timestamp(y, mm, dd)

    return pd.Timestamp(fp.stat().st_mtime, unit="s").normalize()

def _parse_month_token(tok: str) -> pd.Timestamp | None:
    s = tok.strip().upper().replace("-", " ").replace("/", " ")
    parts = s.split()
    if len(parts) == 2:
        a, b = parts
        if a.isdigit() and len(a) == 4 and b in MONTH_MAP:
            return pd.Timestamp(int(a), MONTH_MAP[b], 1)
        if b.isdigit() and len(b) == 4 and a[:3] in MONTH_MAP:
            return pd.Timestamp(int(b), MONTH_MAP[a[:3]], 1)
    try:
        d = dtparse(tok, yearfirst=True, dayfirst=False).date()
        return pd.Timestamp(d.year, d.month, 1)
    except Exception:
        return None

def parse_single_csv(csv_path: Path) -> pd.DataFrame:
    header, data_start, has_header = _detect_header_and_data_start(csv_path)

    if has_header:
        df = pd.read_csv(csv_path, skiprows=data_start)
        cols_lower = {c.lower(): c for c in df.columns}
        date_col = cols_lower.get("date") or cols_lower.get("time") or cols_lower.get("period") or df.columns[0]
        value_col = (
            cols_lower.get("value")
            or cols_lower.get("values")
            or cols_lower.get("observation")
            or (df.columns[1] if len(df.columns) > 1 else df.columns[0])
        )
    else:
        df = pd.read_csv(csv_path, skiprows=data_start, header=None, names=["period", "value"])
        date_col, value_col = "period", "value"

    df = df[[date_col, value_col]].rename(columns={date_col: "period", value_col: "value"})
    df["obs_date"] = df["period"].astype(str).map(_parse_month_token)

    df["value"] = (
        df["value"].astype(str).str.replace(",", "", regex=False).pipe(pd.to_numeric, errors="coerce")
    )

    df = (
        df.dropna(subset=["obs_date", "value"])
          .loc[:, ["obs_date", "value"]]
          .sort_values("obs_date")
          .reset_index(drop=True)
    )

    vintage_date = _extract_vintage_date(header, csv_path)
    df.insert(1, "vintage_date", vintage_date)
    return df[["obs_date", "vintage_date", "value"]]

def build_tidy_vintages(csv_files: list[Path]) -> pd.DataFrame:
    parts = [parse_single_csv(fp) for fp in csv_files]  
    return (
        pd.concat(parts, ignore_index=True)
          .sort_values(["obs_date", "vintage_date"])
          .drop_duplicates(subset=["obs_date", "vintage_date"])
          .reset_index(drop=True)
    )

# run consolidation 
files = sorted(RAW_DIR.glob("AP2Y_prev_*.csv"))  
tidy = build_tidy_vintages(files)

out_csv = PROCESSED_DIR / "vacancies_vintages.csv"
tidy.to_csv(out_csv, index=False)

print(f"Saved: {out_csv}")
print(f"Rows: {len(tidy):,} | Vintages: {tidy['vintage_date'].nunique()} | Obs months: {tidy['obs_date'].nunique()}")


In [ ]:
# 2b. Quick spot check 

print("\n=== Head (first 8 rows) ===")
print(tidy.head(8).to_string(index=False))

print("\n=== Tail (last 8 rows) ===")
print(tidy.tail(8).to_string(index=False))

print("\n=== Latest 10 rows (by vintage_date then obs_date, both desc) ===")
print(
    tidy.sort_values(["vintage_date", "obs_date"], ascending=[False, False])
        .head(10)
        .to_string(index=False)
)

print("\n=== Vintages summary (latest 5) — distinct obs months per vintage ===")
print(
    tidy.groupby("vintage_date")["obs_date"]
        .nunique()
        .sort_index(ascending=False)
        .head(5)
        .to_string()
)

mono_ok = (
    tidy.sort_values(["vintage_date", "obs_date"])
        .groupby("vintage_date")["obs_date"]
        .apply(lambda s: s.is_monotonic_increasing)
        .all()
)
print(f"\nPer-vintage obs_date monotonically increasing: {mono_ok}")

Challenges 
- Vintage date was falling back to download date - needed to amend original code to ensure the release date is captured correctly 
- Some dates were forced into US format - needed to amend the code to ensure dates were consolidated in a consistent format (e.x. as 12-08 instead of 08-12)

Note
- These functions have been written in a simple, fit for purpose way, and extensive cleaning and checks has been skipped as we are dealing with a small and consistent data set. 
- Additional failsafes should be added later to allow for naming convention and formatting changes over time. 
- Solutions to help with speed/size can also be added later (e.x. parquet) 

# 3. Plot data 

o Visualise vacancy revisions and patterns that occur over time.

In [ ]:
# 3a. Visualise revisions for latest month with revisions

PROCESSED_DIR = (Path.cwd() / "../data/processed").resolve()
tidy = pd.read_csv(PROCESSED_DIR / "vacancies_vintages.csv",
                   parse_dates=["obs_date", "vintage_date"])

def pick_latest_month_with_revisions(df: pd.DataFrame, min_vintages: int = 2) -> pd.Timestamp:
    counts = df.groupby("obs_date")["vintage_date"].nunique()
    eligible = counts[counts >= min_vintages]
    if eligible.empty:
        raise ValueError("No months have the required number of vintages.")
    return eligible.index.max()

def plot_revisions_for_month(df: pd.DataFrame, obs_month: pd.Timestamp | None = None):
    if obs_month is None:
        obs_month = pick_latest_month_with_revisions(df, min_vintages=2)

    rev = (df.loc[df["obs_date"] == obs_month]
             .sort_values("vintage_date")
             .drop_duplicates(subset=["vintage_date"]))

    if rev.empty:
        raise ValueError(f"No data found for obs_month={obs_month}.")

    print("\nRevisions table:")
    print(rev[["vintage_date", "value"]].to_string(index=False))

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(rev["vintage_date"], rev["value"], marker="o", linewidth=1.5)
    ax.set_title(f"Revisions for {obs_month.strftime('%B %Y')} vacancy estimate")
    ax.set_xlabel("Vintage (release date)")
    ax.set_ylabel("Vacancy estimate")
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    fig.autofmt_xdate()

    first, last = rev.iloc[0], rev.iloc[-1]
    delta = last["value"] - first["value"]
    pct = (delta / first["value"] * 100.0) if first["value"] else float("nan")
    ax.annotate(f"First: {first['value']:.0f}",
                xy=(first["vintage_date"], first["value"]), xytext=(0, 10),
                textcoords="offset points")
    ax.annotate(f"Latest: {last['value']:.0f}  ({delta:+.0f}, {pct:+.1f}%)",
                xy=(last["vintage_date"], last["value"]), xytext=(0, 10),
                textcoords="offset points", ha="center")

    plt.tight_layout()
    return obs_month, rev

month_used, rev_df = plot_revisions_for_month(tidy)

- Latest month with revisions is very recent (06-2025), and so it is present in only two vintages. 

In [ ]:
# 3b. Check how many months have had revisions 

tidy = pd.read_csv(PROCESSED_DIR / "vacancies_vintages.csv",
                   parse_dates=["obs_date", "vintage_date"])

g = tidy.sort_values(["obs_date","vintage_date"]).groupby("obs_date", as_index=False)

summary = (
    g.agg(
        n_vintages=("vintage_date","nunique"),
        n_distinct_values=("value","nunique"),
        first_value=("value","first"),
        latest_value=("value","last"),
        first_vintage=("vintage_date","first"),
        latest_vintage=("vintage_date","last"),
    )
    .assign(
        delta_abs=lambda d: d["latest_value"] - d["first_value"],
        delta_pct=lambda d: np.where(d["first_value"].ne(0),
                                     100*(d["latest_value"]-d["first_value"])/d["first_value"],
                                     np.nan),
        revised=lambda d: d["n_distinct_values"] > 1
    )
)

print("Overall months in table:", len(summary))
print("Months with ≥2 vintages:", (summary["n_vintages"] >= 2).sum())
print("Months with any revision (value changed):", summary["revised"].sum())

print("\nTop 10 revised months by number of vintages:")
print(summary.loc[summary["revised"]]
             .sort_values(["n_vintages","obs_date"], ascending=[False, True])
             .head(10)[["obs_date","n_vintages","n_distinct_values","delta_abs","delta_pct"]]
             .to_string(index=False))

print("\nTop 10 revised months by absolute change:")
print(summary.loc[summary["revised"]]
             .sort_values(["delta_abs","obs_date"], ascending=[False, True])
             .head(10)[["obs_date","n_vintages","n_distinct_values","delta_abs","delta_pct"]]
             .to_string(index=False))

print("\nLatest 10 months (for context):")
print(summary.sort_values("obs_date").tail(10)[
      ["obs_date","n_vintages","n_distinct_values","delta_abs","delta_pct"]]
      .to_string(index=False))

- Revisions are present for 199/291 months.

- The largest revision is noted in 10-2024 (delta_abs=45).

In [ ]:
# 3c. Visualise revisions for a month with longest revision trail

summary = (
    tidy.sort_values(["obs_date","vintage_date"])
        .groupby("obs_date", as_index=False)
        .agg(n_vintages=("vintage_date","nunique"),
             n_distinct_values=("value","nunique"))
)

revised = summary[summary["n_distinct_values"] > 1]
if revised.empty:
    raise ValueError("No months show revisions within the downloaded vintages.")

max_tail = revised["n_vintages"].max()
month_to_plot = revised.loc[revised["n_vintages"] == max_tail, "obs_date"].max()

print(f"Selected month (longest revision tail): {month_to_plot.date()} "
      f"(vintages: {int(max_tail)})")

_ = plot_revisions_for_month(tidy, month_to_plot)

- December 2023 is a good example of revisions occuring over time - with 5 revisions published between 02-2024 and 09-2025 (note: we have only downloaded 20 vintages).

- Note: ONS website advises that statistics are revised as i. estimates become updated, or ii. due to methods/systems changes.

In [ ]:
# 3d. Highlight patterns in vacancies ahead of forecasting (last 10 years data from latest vintage)

PROCESSED_DIR = (Path.cwd() / "../data/processed").resolve()
tidy = pd.read_csv(PROCESSED_DIR / "vacancies_vintages.csv",
                   parse_dates=["obs_date","vintage_date"])

latest = (tidy.sort_values(["obs_date","vintage_date"])
               .groupby("obs_date", as_index=False)
               .tail(1)[["obs_date","value"]])  
s = (latest.set_index("obs_date")
            .sort_index()
            .asfreq("MS"))  

end = s.index.max()
start = end - pd.DateOffset(years=10) + pd.offsets.MonthBegin(0)
s10 = s.loc[start:end].copy()
y = s10["value"].astype(float)

stl = STL(y, period=12, robust=True)
res = stl.fit()
trend = res.trend
seasonal = res.seasonal
remainder = res.resid
sa = y - seasonal  

fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
axes[0].plot(y.index, trend)      
axes[0].set_title("Trend")
axes[1].plot(y.index, seasonal)
axes[1].set_title("Seasonality")
axes[2].plot(y.index, remainder)
axes[2].set_title("Remainder")
axes[2].set_xlabel("Date")
fig.suptitle("Vacancies — STL decomposition (latest vintage, last 10 years)", y=0.98)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 3.5))
plt.plot(sa.index, sa, linewidth=1.6)
plt.title("Vacancies — seasonally adjusted (latest vintage, last 10 years)")
plt.xlabel("Date")
plt.ylabel("Vacancies (SA)")
plt.tight_layout()
plt.show()

# additional stats
first, last = y.iloc[0], y.iloc[-1]
n_years = max(1, (y.index[-1].year - y.index[0].year) + (y.index[-1].month - y.index[0].month)/12)
cagr = (last/first)**(1/n_years) - 1 if first else np.nan

trend_12m = np.nan
if len(trend) > 12 and trend.iloc[-13] != 0:
    trend_12m = (trend.iloc[-1] - trend.iloc[-13]) / trend.iloc[-13]

m_m = np.nan
if len(sa) > 1 and sa.iloc[-2] != 0:
    m_m = (sa.iloc[-1] - sa.iloc[-2]) / sa.iloc[-2]

y_y = np.nan
if len(y) > 12 and y.iloc[-13] != 0:
    y_y = (y.iloc[-1] - y.iloc[-13]) / y.iloc[-13]

season_tbl = (seasonal.groupby(seasonal.index.month)
                        .mean()
                        .rename(index={1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
                                       7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}))

print("=== Summary (latest vintage, last 10 years) ===")
print(f"CAGR (trend, approx): {cagr*100:,.2f}%")
print(f"Trend change over last 12 months: {trend_12m*100:,.2f}%")
print(f"Seasonally adjusted MoM (last): {m_m*100:,.2f}%")
print(f"Raw YoY (last): {y_y*100:,.2f}%")
print("\nAverage seasonal effect by month (levels):")
print(season_tbl.round(2).to_string())

Observations

- Post pandemic peak in 2022 followed by sustained decline (net flat over a decade)

- Mild seasonality, driven mainly by trend.

- Recent momentum remains negative (SA -0.35% m/m, -14% y/y).

- Seasonal adjustment shows that decline is structural, not a blip from seasonality / random noise. 

    (note: last 10 years only, units in thousands, seasonal effects in levels not %)

Note

- Using decomposition is useful for visualising trend and seasonality ahead of modelling. More accurate and readable visualisations should be generated if required for deeper analysis. 

# 4. Forecast data 

o Build a simple model to forecast future vacancy levels

In [ ]:
# 4a. Forecast next 12 months with STL + ARIMA(1,1,1) - last 10 years data from latest vintage

PROCESSED_DIR = (Path.cwd() / "../data/processed").resolve()
tidy = pd.read_csv(PROCESSED_DIR / "vacancies_vintages.csv",
                   parse_dates=["obs_date","vintage_date"])

latest = (tidy.sort_values(["obs_date","vintage_date"])
               .groupby("obs_date", as_index=False)
               .tail(1)[["obs_date","value"]])
y = (latest.set_index("obs_date")
           .sort_index()
           .asfreq("MS"))["value"].astype(float)

end = y.index.max()
start = end - pd.DateOffset(years=10) + pd.offsets.MonthBegin(0)
y_train = y.loc[start:end].copy()

H = 12
stlf = STLForecast(
    endog=y_train,
    model=ARIMA,
    model_kwargs={"order": (1, 1, 1), "trend": "n"},  
    period=12,
    robust=True,
)
res = stlf.fit()
fc = res.forecast(H)                      
ci = res.get_prediction(start=y_train.index[-1] + pd.offsets.MonthBegin(1),
                        end=y_train.index[-1] + pd.DateOffset(months=H)).conf_int()

print("=== Last 6 actuals + 12-month forecast (STL+ARIMA(1,1,1)) ===")
print(pd.concat([y_train.tail(6).rename("actual"), fc.rename("forecast")], axis=1).to_string())

plt.figure(figsize=(8,4))
hist = y.loc[y.index >= (y.index.max() - pd.DateOffset(years=10))]
plt.plot(hist.index, hist.values, linewidth=1.6, label="Actual (latest vintage)")
plt.plot(fc.index, fc.values, linestyle="--", linewidth=1.6, label="Forecast")
plt.fill_between(ci.index, ci.iloc[:,0], ci.iloc[:,1], alpha=0.2, label="≈95% band")
plt.title("UK vacancies — STL + ARIMA(1,1,1) forecast (12 months)")
plt.xlabel("Date")
plt.ylabel("Vacancies (thousands)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 4b. Forecast next 12 months with Holt-Winters - last 10 years data from latest vintage

PROCESSED_DIR = (Path.cwd() / "../data/processed").resolve()
tidy = pd.read_csv(PROCESSED_DIR / "vacancies_vintages.csv",
                   parse_dates=["obs_date","vintage_date"])

latest = (tidy.sort_values(["obs_date","vintage_date"])
               .groupby("obs_date", as_index=False)
               .tail(1)[["obs_date","value"]])
y = (latest.set_index("obs_date")
           .sort_index()
           .asfreq("MS"))["value"].astype(float)

end = y.index.max()
start = end - pd.DateOffset(years=10) + pd.offsets.MonthBegin(0)
y_train = y.loc[start:end].copy()

model = ExponentialSmoothing(
    y_train,
    trend="add",
    damped_trend=True,
    seasonal="add",
    seasonal_periods=12,
    initialization_method="estimated",
)
fit_hw = model.fit(optimized=True, use_brute=True)

H = 12
fc_hw = fit_hw.forecast(H)

sigma = fit_hw.resid.std(ddof=1)
lo_hw = fc_hw - 1.96 * sigma
hi_hw = fc_hw + 1.96 * sigma

print("=== Last 6 actuals + 12-month forecast (Holt-Winters) ===")
print(pd.concat([y_train.tail(6).rename("actual"), fc_hw.rename("forecast")], axis=1).to_string())

plt.figure(figsize=(8,4))
hist = y.loc[y.index >= (y.index.max() - pd.DateOffset(years=10))]
plt.plot(hist.index, hist.values, linewidth=1.6, label="Actual (latest vintage)")
plt.plot(fc_hw.index, fc_hw.values, linestyle="--", linewidth=1.6, label="Holt-Winters forecast")
plt.fill_between(fc_hw.index, lo_hw.values, hi_hw.values, alpha=0.18, label="HW ≈95% band")
try:
    plt.plot(fc.index, fc.values, linestyle="--", linewidth=1.6, label="STL+ARIMA forecast")
except Exception:
    pass
plt.title("UK vacancies — Holt-Winters baseline vs history (last 10 years)")
plt.xlabel("Date")
plt.ylabel("Vacancies (thousands)")
plt.legend()
plt.tight_layout()
plt.show()

- Both models follow a similar central path, with Holt-Winters a touch higher initially then STL+ARIMA, then drifitng down similarly. 

- Models diverge in autum 2025 - STL+ARIMA adds a small seasonal bump to a modest SA uptick (considering recent momentum is up, Oct usually a bit high); whereas Holt–Winters carries a damped down-trend plus a slightly negative seasonal index for that month (thinking trend is still down, and recent Octs a bit weak). 

- Both models are reasonable and this comparison shows how they weight the evidence differently. A useful thing to keep in mind for further modelling.

In [ ]:
# 4c. Quick evaluation of classical models vs seasonal-naïve 

PROCESSED_DIR = (Path.cwd() / "../data/processed").resolve()
tidy = pd.read_csv(PROCESSED_DIR / "vacancies_vintages.csv",
                   parse_dates=["obs_date","vintage_date"])

latest = (tidy.sort_values(["obs_date","vintage_date"])
               .groupby("obs_date", as_index=False)
               .tail(1)[["obs_date","value"]])
y = (latest.set_index("obs_date").sort_index().asfreq("MS"))["value"].astype(float)

m = 12   
train_years = 10
steps = 12  
errs_hw, errs_stl, errs_sn = [], [], []

def seasonal_scale(insample: pd.Series, m: int) -> float:
    diffs = (insample.iloc[m:] - insample.shift(m).iloc[m:]).abs()
    return float(diffs.mean()) if len(diffs) else np.nan

for h in range(steps, 0, -1):
    y_end = y.index.max() - pd.DateOffset(months=h)
    y_next = y_end + pd.offsets.MonthBegin(1)
    if y_next not in y.index:
        continue

    y_train = y.loc[y_end - pd.DateOffset(years=train_years) + pd.offsets.MonthBegin(0): y_end].dropna()
    if len(y_train) < 36: 
        continue

    fit_hw = ExponentialSmoothing(y_train, trend="add", damped_trend=True,
                                  seasonal="add", seasonal_periods=m,
                                  initialization_method="estimated").fit(optimized=True, use_brute=True)
    pred_hw = float(fit_hw.forecast(1).iloc[0])

    fit_stl = STLForecast(y_train, model=ARIMA, model_kwargs={"order":(1,1,1), "trend":"n"},
                          period=m, robust=True).fit()
    pred_stl = float(fit_stl.forecast(1).iloc[0])

    t_minus_m = y_next - pd.DateOffset(months=m)
    if t_minus_m not in y.index:
        continue
    pred_sn = float(y.loc[t_minus_m])

    y_true = float(y.loc[y_next])
    errs_hw.append(abs(y_true - pred_hw))
    errs_stl.append(abs(y_true - pred_stl))
    errs_sn.append(abs(y_true - pred_sn))

mae_hw  = float(np.mean(errs_hw))  if errs_hw  else np.nan
mae_stl = float(np.mean(errs_stl)) if errs_stl else np.nan
mae_sn  = float(np.mean(errs_sn))  if errs_sn  else np.nan

scale = seasonal_scale(y.loc[y.index <= y.index.max() - pd.DateOffset(months=steps)], m)
mase_hw  = mae_hw  / scale if scale else np.nan
mase_stl = mae_stl / scale if scale else np.nan
mase_sn  = mae_sn  / scale if scale else np.nan  

res = pd.DataFrame(
    {"Model": ["STL+ARIMA(1,1,1)","Holt-Winters (A,Adamp,A)","Seasonal-Naïve"],
     "MAE":   [mae_stl, mae_hw, mae_sn],
     "MASE":  [mase_stl, mase_hw, mase_sn]}
).set_index("Model").round(2)

print("=== One-step-ahead over last 12 months ===")
print(res.to_string())
print("\nInterpretation: MASE < 1 beats seasonal-naïve. Lower is better.")

- Holt-Winters is the best baseline to carry forward, as it beats STL+ARIMA and Seasonal-Naive (MASE 0.21 vs 0.28, and 1.22 respectively). 
- Note that this was just a quick sanity check and a proper evaluation should be vintage-aware (rolling real-time backtest vs first-published values).
- To re-evaluate in real time - for each recent release, refit the model on the as-of-vintage data and score against both first-published and revised values, then note typical revision drift so error sizes are interpreted in that context. 

# 5. Summary and recommendations

The analysis of historical vacancy data successfully identified key time-based patterns in vacancies which contributed to developing a simple forecasting model to support decision-making for monetary policy. Guided by the the principle to always start with a simple model and gradually increase complexity only if necessary, a forecast baseline model was created using classical time series methods that captured mild seasonal trends. 

It would be valuable to compare these results to a machine learning model as they can provide additional insights, can handle volatility and external influences. A decision tree ensemble, like XGBoost would be a reliable choice, or a deep learning neural network like LSTM (if features grow) - while a hybrid solution of both could increase forecast accuracy even further. It is crucial to consult the stakeholders to understand the desired balance between acceptable results, and the required development effort and computation power. 

Ultimately, to make the forecasts truly powerful, we should build models that take into account external factors rather than just relying on historical data, ones that also allow for scenario planning and establishing causality. For the final forecast pipeline, the data scraping and consolidation code must be optimised for speed and future proofed against varying data, with preprocessing code ensuring thorough cleaning and minimal noise going into the forecasts. This will ensure that the forecasts can be trusted and are genuinely useful. 